# ResNet50 Fine-Tuning — BUS-BRA (Fixed Version)

In [ ]:
import os, json, random, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from PIL import Image
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import AUC, Recall, Precision
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

SEED = 42
tf.random.set_seed(SEED); np.random.seed(SEED); random.seed(SEED)
print('TF:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
import gdown

file_id = '1GudAb2jP665CUsCX8q1HWbZGFPeoJpet' # ID 
url = f'https://drive.google.com/uc?id={file_id}'

output_path = '/kaggle/working/base_model.keras' 

gdown.download(url, output_path, quiet=False)

print(f" : {output_path}")

In [ ]:
# =====================================================================
TRAIN_CSV = '/kaggle/input/datasets/habibashhefny/ulatrasound-split-test/train_predictions (1).csv'
VAL_CSV = '/kaggle/input/datasets/habibashhefny/ulatrasound-split-test/validation_predictions (1).csv'
TEST_CSV = '/kaggle/input/datasets/habibashhefny/ulatrasound-split-test/test_predictions (1).csv'
MODEL_PATH = '/kaggle/working/base_model.keras'
IMAGES_DIR='/kaggle/input/datasets/orvile/bus-bra-a-breast-ultrasound-dataset/BUSBRA/BUSBRA/Images'
EXTRACT_TO = '/kaggle/working/Images'
SAVE_DIR = '/kaggle/working/output'
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE = 224
CLASS_NAMES = ['benign', 'malignant']
NUM_CLASSES = 2 
BATCH_SIZE = 16
PHASE1_LR = 1e-4; PHASE1_EPOCHS = 15
PHASE2_LR = 1e-5; PHASE2_EPOCHS = 25
UNFREEZE_LAST = 30
# =====================================================================
print('Config ready ')

In [ ]:
def load_and_fix(csv_path, images_dir):
 df = pd.read_csv(csv_path)
 df['image_path'] = df['image_path'].apply(
 lambda p: os.path.join(images_dir, os.path.basename(p))
 )
 missing = df['image_path'].apply(lambda p: not os.path.exists(p)).sum()
 return df, missing

train_df, m1 = load_and_fix(TRAIN_CSV, IMAGES_DIR)
val_df, m2 = load_and_fix(VAL_CSV, IMAGES_DIR)
test_df, m3 = load_and_fix(TEST_CSV, IMAGES_DIR)

print('CSVs loaded ')
for name, df, miss in [('train', train_df, m1), ('val', val_df, m2), ('test', test_df, m3)]:
 print(f' {name:<6}: {len(df):>5} rows | '
 f'benign={df["label"].eq(0).sum()} | '
 f'malignant={df["label"].eq(1).sum()} | '
 f'missing={miss}')

if m1 + m2 + m3 > 0:
 raise FileNotFoundError(' ! IMAGES_DIR')

print()
print('Test set status: PROTECTED - will not be used during training')

In [ ]:
# Preprocessing — 
def preprocess_ultrasound_image(image_path, img_size=IMG_SIZE):
 img = Image.open(image_path).convert('RGB')
 img = img.resize((img_size, img_size))
 arr = np.array(img, dtype=np.float32)
 arr = np.expand_dims(arr, axis=0)
 arr = preprocess_input(arr)
 return arr # shape: (1, 224, 224, 3)


def load_sample(image_path, label):
 def _load(path, lbl):
 arr = preprocess_ultrasound_image(path.numpy().decode('utf-8'))
 arr = arr[0]
 lbl = tf.keras.utils.to_categorical(lbl.numpy(), num_classes=NUM_CLASSES)
 return arr.astype(np.float32), lbl.astype(np.float32)
 img, lbl = tf.py_function(_load, [image_path, label], [tf.float32, tf.float32])
 img.set_shape((IMG_SIZE, IMG_SIZE, 3))
 lbl.set_shape((NUM_CLASSES,))
 return img, lbl


def make_dataset(df, shuffle=False):
 ds = tf.data.Dataset.from_tensor_slices(
 (df['image_path'].values, df['label'].values.astype(np.int32))
 )
 if shuffle:
 ds = ds.shuffle(len(df), seed=SEED)
 return (ds
 .map(load_sample, num_parallel_calls=tf.data.AUTOTUNE)
 .batch(BATCH_SIZE)
 .prefetch(tf.data.AUTOTUNE))


train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)
test_ds = make_dataset(test_df, shuffle=False)

print('Datasets ready ')
print(f' train : {len(train_df)} imgs → {len(train_ds)} batches')
print(f' val : {len(val_df)} imgs → {len(val_ds)} batches')
print(f' test : {len(test_df)} imgs → {len(test_ds)} batches (unseen)')

In [ ]:
# Class Weights
cw = compute_class_weight('balanced', classes=np.array([0,1]), y=train_df['label'].values)
class_weights = {0: cw[0], 1: cw[1]}
print('Class weights:', {CLASS_NAMES[k]: round(v, 3) for k, v in class_weights.items()})

## Fine-tune Head

** :**
 **3 outputs** (benign/malignant/normal) dataset normal cases.
 CSVs **2 classes ** (benign=0, malignant=1).

**:**
 base (ResNet50 backbone) **head 2 outputs**.

In [ ]:
# ── (3 outputs) ────────────────────────────────
print('Loading pretrained model...')
pretrained_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print(' Input shape:', pretrained_model.input_shape)
print(' Output shape:', pretrained_model.output_shape)
print(' Total layers:', len(pretrained_model.layers))

# ── ResNet50 backbone ────────────────────────────────────
backbone = None
for layer in pretrained_model.layers:
 if isinstance(layer, tf.keras.Model) or 'resnet50' in layer.name.lower():
 backbone = layer
 print(f' Backbone found: {layer.name}')
 break

if backbone is None:
 # fallback: 3 layers (GAP + Dense + Softmax)
 backbone_input = pretrained_model.input
 backbone_output = pretrained_model.layers[-4].output
 print(' Backbone extracted from layers (fallback method)')

In [ ]:
# ── Fine-tune Model 2 outputs ────────────────────────────
if backbone is not None:
 backbone.trainable = False
 inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input')
 x = backbone(inputs, training=False)
 x = layers.GlobalAveragePooling2D(name='gap')(x)
 x = layers.Dense(256, activation='relu', name='fc1')(x)
 x = layers.Dropout(0.4, name='drop')(x)
 outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='out')(x) # 2 outputs
else:
 # fallback
 inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input')
 x = tf.keras.Model(inputs=backbone_input, outputs=backbone_output)(inputs)
 x = layers.GlobalAveragePooling2D(name='gap')(x)
 x = layers.Dense(256, activation='relu', name='fc1')(x)
 x = layers.Dropout(0.4, name='drop')(x)
 outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='out')(x)

ft_model = Model(inputs, outputs, name='ResNet50_BUSBRA_FT')
ft_model.compile(
 optimizer=tf.keras.optimizers.Adam(PHASE1_LR),
 loss='categorical_crossentropy',
 metrics=['accuracy', AUC(name='auc'), Recall(name='recall'), Precision(name='precision')]
)

trainable_params = sum(np.prod(v.shape) for v in ft_model.trainable_variables)
total_params = sum(np.prod(v.shape) for v in ft_model.variables)
print(f'Fine-tune model ready ')
print(f' Trainable: {trainable_params:,} / {total_params:,} params (head only)')
print(f' Output shape: {ft_model.output_shape} (should be (None, 2))')

In [ ]:
def make_callbacks(phase):
 return [
 EarlyStopping(
 monitor='val_auc', patience=6,
 restore_best_weights=True, mode='max', verbose=1
 ),
 ReduceLROnPlateau(
 monitor='val_loss', factor=0.3,
 patience=3, min_lr=1e-7, verbose=1
 ),
 ModelCheckpoint(
 os.path.join(SAVE_DIR, f'best_{phase}.keras'),
 monitor='val_auc', save_best_only=True, mode='max', verbose=1
 )
 ]
print('Callbacks ready ')

## Phase 1 — Head Training (Backbone Frozen)

 train head — backbone .

In [ ]:
print('=' * 55)
print(' PHASE 1: Head only — backbone frozen')
print('=' * 55)

history_p1 = ft_model.fit(
 train_ds,
 validation_data=val_ds, # val — test 
 epochs=PHASE1_EPOCHS,
 callbacks=make_callbacks('phase1'),
 class_weight=class_weights,
 verbose=1
)

print(f'Best val_auc P1: {max(history_p1.history["val_auc"]):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Phase 1 — Head Training', fontsize=13, fontweight='bold')
for ax, m, t in zip(axes, ['loss','accuracy','auc'], ['Loss','Accuracy','AUC']):
 ax.plot(history_p1.history[m], label='Train', marker='o', ms=4)
 ax.plot(history_p1.history[f'val_{m}'], label='Val', marker='s', ms=4)
 ax.set_title(t); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'phase1_curves.png'), dpi=150)
plt.show()

## Phase 2 — Fine-Tuning (Unfreeze Last 30 Layers)

 30 layer backbone train learning rate .

In [ ]:
print('=' * 55)
print(f' PHASE 2: Unfreeze last {UNFREEZE_LAST} layers')
print('=' * 55)

if backbone is not None:
 backbone.trainable = True
 for layer in backbone.layers[:-UNFREEZE_LAST]:
 layer.trainable = False
 unfrozen = sum(1 for l in backbone.layers if l.trainable)
 print(f' Unfrozen: {unfrozen} / {len(backbone.layers)} layers')
else:
 # fallback: unfreeze last 30 layers of the whole model
 for layer in ft_model.layers[:-UNFREEZE_LAST]:
 layer.trainable = False
 for layer in ft_model.layers[-UNFREEZE_LAST:]:
 layer.trainable = True
 print(f' Unfrozen: last {UNFREEZE_LAST} layers of ft_model')

ft_model.compile(
 optimizer=tf.keras.optimizers.Adam(PHASE2_LR),
 loss='categorical_crossentropy',
 metrics=['accuracy', AUC(name='auc'), Recall(name='recall'), Precision(name='precision')]
)

history_p2 = ft_model.fit(
 train_ds,
 validation_data=val_ds, # val — test 
 epochs=PHASE2_EPOCHS,
 callbacks=make_callbacks('phase2'),
 class_weight=class_weights,
 verbose=1
)

print(f'Best val_auc P2: {max(history_p2.history["val_auc"]):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Phase 2 — Fine-Tuning', fontsize=13, fontweight='bold')
for ax, m, t in zip(axes, ['loss','accuracy','auc'], ['Loss','Accuracy','AUC']):
 ax.plot(history_p2.history[m], label='Train', marker='o', ms=4)
 ax.plot(history_p2.history[f'val_{m}'], label='Val', marker='s', ms=4)
 ax.set_title(t); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'phase2_curves.png'), dpi=150)
plt.show()

## 

** :**
1. Validation set — 
2. Test set (unseen) — 

In [ ]:
def evaluate_split(ds, df, split_name):
 """
 : classification report + confusion matrix + AUC + PS extraction
 """
 y_true, y_pred, y_prob = [], [], []

 for x_batch, y_batch in ds:
 preds = ft_model.predict(x_batch, verbose=0)
 y_true.extend(np.argmax(y_batch.numpy(), axis=1))
 y_pred.extend(np.argmax(preds, axis=1))
 y_prob.extend(preds[:, 1]) # P(malignant)

 y_true = np.array(y_true)
 y_pred = np.array(y_pred)
 y_prob = np.array(y_prob)

 auc = roc_auc_score(y_true, y_prob)

 print(f'\n── {split_name} ─────────────────────────────')
 print(f'AUC: {auc:.4f}')
 print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

 cm = confusion_matrix(y_true, y_pred)
 fig, ax = plt.subplots(figsize=(5, 4))
 sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
 xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
 ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
 ax.set_title(f'Confusion Matrix — {split_name}')
 plt.tight_layout()
 plt.savefig(os.path.join(SAVE_DIR, f'cm_{split_name.lower().replace(" ","_")}.png'), dpi=150)
 plt.show()

 return y_true, y_pred, y_prob, auc


# ── Validation ─────────────────────────────────────────────────────────
y_true_val, y_pred_val, y_prob_val, auc_val = evaluate_split(val_ds, val_df, 'Validation')

In [ ]:
# ── Test Set (Unseen) — ───────────────────────
print('Evaluating on UNSEEN test set...')
print('This set was never used during training or validation.')

y_true_test, y_pred_test, y_prob_test, auc_test = evaluate_split(test_ds, test_df, 'Test (Unseen)')

print(f'\nSUMMARY:')
print(f' Val AUC: {auc_val:.4f}')
print(f' Test AUC: {auc_test:.4f}')